# Study 1: Full Root Cause Experiment
**3 probes × 3 variants × 50 items × 6 models × 3 repeats = 8,100 judgments**

Free GPU (P100) · ~8-10 hours runtime

In [ ]:
!pip install -q transformers torch accelerate huggingface_hub pandas numpy matplotlib scipy
import torch, gc, os, json, re, time, random
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import getpass
random.seed(42)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
token = getpass.getpass('HuggingFace token: ')
login(token=token)
os.environ['HF_TOKEN'] = token

MODELS = {
    "llama3-base": "meta-llama/Meta-Llama-3-8B",
    "llama3-inst": "meta-llama/Meta-Llama-3-8B-Instruct",
    "mistral-base": "mistralai/Mistral-7B-v0.3",
    "mistral-inst": "mistralai/Mistral-7B-Instruct-v0.3",
    "gemma2-base": "google/gemma-2-2b",
    "gemma2-inst": "google/gemma-2-2b-it",
}
print(f"Registered {len(MODELS)} models")

In [ ]:
# 50 evaluation items
ITEMS = [
    {"instr": "Explain how photosynthesis works.", "resp": "Plants use sunlight to convert CO2 and water into glucose and oxygen. Chlorophyll captures light energy."},
    {"instr": "What is the theory of relativity?", "resp": "Einstein's theory says that space and time are relative to the observer's frame of reference."},
    {"instr": "Describe the water cycle.", "resp": "Water evaporates from oceans, forms clouds through condensation, and returns as precipitation."},
    {"instr": "What causes earthquakes?", "resp": "Tectonic plates shift along fault lines, releasing energy as seismic waves."},
    {"instr": "Explain how vaccines work.", "resp": "Vaccines train the immune system to recognize pathogens by introducing harmless antigens."},
    {"instr": "What is DNA?", "resp": "DNA is a molecule that carries genetic instructions for growth, development, and reproduction."},
    {"instr": "Describe the solar system.", "resp": "The solar system has 8 planets orbiting the Sun, with Earth being the third from the Sun."},
    {"instr": "What is entropy?", "resp": "Entropy measures disorder in a system. The second law says entropy always increases."},
    {"instr": "How do batteries work?", "resp": "Chemical reactions create electron flow between electrodes through an electrolyte."},
    {"instr": "What is a black hole?", "resp": "A region where gravity is so strong that nothing, not even light, can escape."},
    {"instr": "What is machine learning?", "resp": "A subset of AI where systems learn patterns from data without explicit programming."},
    {"instr": "Describe cloud computing.", "resp": "On-demand computing resources delivered over the internet, including storage and processing."},
    {"instr": "What is an API?", "resp": "An interface that allows different software applications to communicate with each other."},
    {"instr": "Explain how encryption works.", "resp": "Algorithms transform readable data into encoded form using cryptographic keys."},
    {"instr": "What is a database index?", "resp": "A data structure that speeds up data retrieval, like a book's index."},
    {"instr": "What is Python?", "resp": "A high-level programming language known for readability and extensive libraries."},
    {"instr": "Explain the internet.", "resp": "A global network of computers communicating via standardized protocols like TCP/IP."},
    {"instr": "What is a blockchain?", "resp": "A distributed ledger where transactions are recorded in immutable blocks."},
    {"instr": "What is an operating system?", "resp": "Software managing hardware and providing services for application programs."},
    {"instr": "Explain neural networks.", "resp": "Computing systems inspired by biological neurons that learn from labeled data."},
    {"instr": "What caused World War 1?", "resp": "Assassination of Archduke Ferdinand triggered alliances, nationalism, and militarism."},
    {"instr": "Explain democracy.", "resp": "A system where citizens vote for representatives to make decisions on their behalf."},
    {"instr": "What is the Renaissance?", "resp": "A cultural rebirth in Europe from the 14th to 17th century, reviving art and learning."},
    {"instr": "Describe capitalism.", "resp": "An economic system with private ownership, profit motive, and market competition."},
    {"instr": "What is the UN?", "resp": "An international organization promoting peace, security, and cooperation among nations."},
    {"instr": "Explain the Cold War.", "resp": "Geopolitical tension between US and USSR from 1947 to 1991 without direct conflict."},
    {"instr": "What is ethics?", "resp": "The study of moral principles guiding right and wrong behavior."},
    {"instr": "Describe feudalism.", "resp": "A medieval hierarchy where lords granted land to vassals in exchange for service."},
    {"instr": "What is philosophy?", "resp": "The study of fundamental questions about existence, knowledge, and values."},
    {"instr": "Explain globalization.", "resp": "Increasing interconnection of economies, cultures, and populations worldwide."},
    {"instr": "How to cook pasta?", "resp": "Boil water, add salt, cook pasta until al dente, drain, and serve with sauce."},
    {"instr": "What is a healthy diet?", "resp": "Balanced meals with fruits, vegetables, protein, whole grains, and limited processed foods."},
    {"instr": "Explain how cars work.", "resp": "Engine burns fuel to create combustion that turns pistons and rotates the wheels."},
    {"instr": "How to exercise properly?", "resp": "Warm up, do strength and cardio exercises with proper form, cool down and stretch."},
    {"instr": "Describe sleep cycles.", "resp": "Sleep alternates between REM and non-REM stages in 90-minute cycles."},
    {"instr": "What is meditation?", "resp": "A practice of focusing attention to achieve mental clarity and emotional calm."},
    {"instr": "Explain first aid.", "resp": "Emergency care including CPR, wound cleaning, and calling for professional help."},
    {"instr": "How to save money?", "resp": "Track expenses, create a budget, reduce unnecessary spending, and save regularly."},
    {"instr": "What is climate change?", "resp": "Long-term shifts in temperature and weather due to greenhouse gas emissions."},
    {"instr": "Explain recycling.", "resp": "Processing waste materials into new products to reduce resource consumption."},
    {"instr": "What is calculus?", "resp": "The mathematical study of change, involving derivatives and integrals."},
    {"instr": "Explain p-value.", "resp": "Probability of observing results as extreme as actual, assuming null hypothesis is true."},
    {"instr": "What is a prime number?", "resp": "A number divisible only by 1 and itself, like 2, 3, 5, 7, 11."},
    {"instr": "Describe standard deviation.", "resp": "A measure of how spread out numbers are from the mean of a dataset."},
    {"instr": "What is a logarithm?", "resp": "The inverse of exponentiation, showing what power a base must be raised to."},
    {"instr": "Explain probability.", "resp": "The measure of likelihood that an event will occur, from 0 to 1."},
    {"instr": "What is Bayes theorem?", "resp": "Describes probability of an event based on prior knowledge of related conditions."},
    {"instr": "Explain linear regression.", "resp": "A method to model the relationship between variables by fitting a linear equation."},
    {"instr": "What is a derivative?", "resp": "The rate at which a function changes at a given point."},
    {"instr": "Describe correlation.", "resp": "A statistical measure of how two variables move together, from -1 to 1."},
]

In [ ]:
# 3 probes with 3 variants each
PROBES = {
    "rubric_order": {
        "normal": "Score the following response from 1 to 5 (1=worst, 5=best).",
        "reversed": "Score the following response from 1 to 5 (1=best, 5=worst).",
        "random": "Rate the following on a scale where 3=A, 5=B, 1=C, 2=D, 4=E.",
    },
    "score_id": {
        "numeric": "Score from 1 to 5.",
        "letter": "Score from A to E (A=best, E=worst).",
        "descriptive": "Score: Poor, Fair, Good, Very Good, Excellent.",
    },
    "reference_answer": {
        "no_ref": "Score the following response from 1 to 5.",
        "good_ref": "Example good answer: AI systems improve through training.\n\nNow score the following from 1 to 5.",
        "poor_ref": "Example poor answer: idk lol.\n\nNow score the following from 1 to 5.",
    },
}

def build_prompt(probe_type, variant, item):
    prefix = PROBES[probe_type][variant]
    return f"{prefix}\n\n### Instruction:\n{item['instr']}\n\n### Response:\n{item['resp']}\n\n### Score:"

def extract_score(text):
    match = re.search(r'\b([1-5])\b', text)
    return int(match.group(1)) if match else 3

def extract_letter(text):
    match = re.search(r'\b([A-E])\b', text.upper())
    return {'A':5,'B':4,'C':3,'D':2,'E':1}.get(match.group(1), 3)

def extract_descriptive(text):
    m = text.lower()
    for word, score in [('excellent',5),('very good',4),('good',4),('fair',3),('poor',2),('terrible',1)]:
        if word in m:
            return score
    return extract_score(text)

def get_scorer(probe_type, variant):
    if probe_type == 'score_id':
        if variant == 'letter': return extract_letter
        if variant == 'descriptive': return extract_descriptive
    return extract_score

def run_probe(model, tokenizer, probe_type, variants, items, repeats):
    results = {}
    for var_name in variants:
        var_scores = []
        for rep in range(repeats):
            rep_scores = []
            for item in items:
                prompt = build_prompt(probe_type, var_name, item)
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
                with torch.no_grad():
                    outputs = model.generate(**inputs, max_new_tokens=3, temperature=0.0, do_sample=False, pad_token_id=tokenizer.pad_token_id)
                out_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
                rep_scores.append(get_scorer(probe_type, var_name)(out_text))
            var_scores.append(rep_scores)
        results[var_name] = var_scores
        avg = sum(sum(r) for r in var_scores) / (len(var_scores)*len(var_scores[0]))
        print(f"    {var_name}: {avg:.2f} avg")
    return results

REPEATS = 3

In [ ]:
all_results = {}
probes_to_run = [
    ("rubric_order", ["normal", "reversed", "random"]),
    ("score_id", ["numeric", "letter", "descriptive"]),
    ("reference_answer", ["no_ref", "good_ref", "poor_ref"]),
]

for model_key, hf_id in MODELS.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {model_key} ({hf_id})")
    print('='*60)
    print(f"Loading...", end=" ", flush=True)
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    model.eval()
    print(f"OK ({time.time()-t0:.0f}s)")
    all_results[model_key] = {}
    for probe_type, variants in probes_to_run:
        print(f"  Probe: {probe_type}")
        all_results[model_key][probe_type] = run_probe(model, tokenizer, probe_type, variants, ITEMS, REPEATS)
    del model, tokenizer; gc.collect(); torch.cuda.empty_cache()
    print(f"  Freed memory.")

In [ ]:
print("\n" + "="*70)
print("STUDY 1: COMPLETE ROOT CAUSE RESULTS")
print("="*70)
print(f"{'Model':<15} {'Probe':<20} {'Control':<10} {'Biased1':<10} {'Biased2':<10} {'Max Delta':<10}")
print("-"*70)
summary = {}
for model_key in MODELS:
    summary[model_key] = {}
    for probe_type, variants in probes_to_run:
        if model_key not in all_results or probe_type not in all_results[model_key]: continue
        res = all_results[model_key][probe_type]
        means = {}
        for v in variants:
            if v in res:
                s = sum(sum(r) for r in res[v]) / sum(len(r) for r in res[v])
                means[v] = s
        if len(means) >= 2:
            control = means.get(variants[0], 0)
            b1 = means.get(variants[1], 0)
            b2 = means.get(variants[2], 0)
            max_d = max(abs(control-b1), abs(control-b2))
            summary[model_key][probe_type] = {'control':control,'biased1':b1,'biased2':b2,'max_delta':max_d}
            print(f"{model_key:<15} {probe_type:<20} {control:<10.3f} {b1:<10.3f} {b2:<10.3f} {max_d:<.3f}")

print("\n" + "="*70)
print("ROOT CAUSE: BASE vs INSTRUCT")
print("="*70)
print(f"{'Family':<15} {'Base Bias':<12} {'Instruct Bias':<15} {'Amp Factor':<15}")
print("-"*55)
for family in ['llama3','mistral','gemma2']:
    b = f"{family}-base"
    i = f"{family}-inst"
    if b in summary and i in summary:
        bb = sum(summary[b].get(p,{}).get('max_delta',0) for p,_,_ in probes_to_run)/3
        bi = sum(summary[i].get(p,{}).get('max_delta',0) for p,_,_ in probes_to_run)/3
        factor = bi / max(bb, 0.001)
        print(f"{family:<15} {bb:<12.3f} {bi:<15.3f} {factor:<.2f}x")

In [ ]:
save_path = "/kaggle/working/study1_complete.json"
with open(save_path, "w") as f:
    json.dump({"all_results": all_results, "summary": summary}, f, indent=2)
print(f"Results: {save_path}")
total = sum(len(all_results.get(m,{}).get(p,{}).get(v,[[]])[0]) for m in MODELS for p,vs in probes_to_run for v in vs if m in all_results and p in all_results[m] and v in all_results[m][p])
print(f"Total judgments: {total}")
print("Download: Kaggle sidebar -> Output -> Data -> study1_complete.json")